## **1. Import libraries**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix

torch.manual_seed(17)

## **2. Read the data**

In [ ]:
x_train = torch.tensor(np.loadtxt("./data/train_data.csv", delimiter=","), dtype=torch.float32)
y_train = torch.tensor(np.loadtxt("./data/train_label.csv", delimiter=","), dtype=torch.long)
x_val = torch.tensor(np.loadtxt("./data/val_data.csv", delimiter=","), dtype=torch.float32)
y_val = torch.tensor(np.loadtxt("./data/val_label.csv", delimiter=","), dtype=torch.long)
x_test = torch.tensor(np.loadtxt("./data/test_data.csv", delimiter=","), dtype=torch.float32)
y_test = torch.tensor(np.loadtxt("./data/test_label.csv", delimiter=","), dtype=torch.long)

x_train.shape, x_val.shape, x_test.shape

## **3. Define the network**

The network is 36-30-20-5 with ReLU hidden layers.

In [ ]:
class Network(nn.Module):
    def __init__(self, dropout=0, dropconnect=0, batch_norm=False):
        super().__init__()
        self.fc1 = nn.Linear(36, 30)
        self.fc2 = nn.Linear(30, 20)
        self.fc3 = nn.Linear(20, 5)
        self.bn1 = nn.BatchNorm1d(30) if batch_norm else nn.Identity()
        self.bn2 = nn.BatchNorm1d(20) if batch_norm else nn.Identity()
        self.dropout = dropout
        self.dropconnect = dropconnect

    def linear(self, x, layer):
        if not self.training or self.dropconnect == 0:
            return layer(x)
        keep = 1 - self.dropconnect
        mask = torch.bernoulli(torch.full_like(layer.weight, keep)) / keep
        return F.linear(x, layer.weight * mask, layer.bias)

    def forward(self, x):
        x = self.bn1(F.relu(self.linear(x, self.fc1)))
        x = F.dropout(x, self.dropout, self.training)
        x = self.bn2(F.relu(self.linear(x, self.fc2)))
        x = F.dropout(x, self.dropout, self.training)
        return self.fc3(x)

## **4. Define training and evaluation**

In [ ]:
def train_model(model, l2=0):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    train_error, val_error = [], []
    generator = torch.Generator().manual_seed(17)
    stable_epochs = 0

    for epoch in range(300):
        model.train()
        indices = torch.randperm(len(x_train), generator=generator)

        for batch in indices.chunk(5):
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x_train[batch]), y_train[batch])
            if l2:
                weight_sum = sum(layer.weight.square().sum() for layer in [model.fc1, model.fc2, model.fc3])
                loss += l2 * weight_sum / (2 * len(x_train))
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            train_error.append(F.cross_entropy(model(x_train), y_train).item())
            val_error.append(F.cross_entropy(model(x_val), y_val).item())

        if epoch >= 19 and abs(train_error[-1] - train_error[-2]) < 0.001:
            stable_epochs += 1
        else:
            stable_epochs = 0
        if stable_epochs == 5:
            break

    return train_error, val_error

In [ ]:
def evaluate(model, features, labels):
    model.eval()
    with torch.no_grad():
        predicted = model(features).argmax(1).numpy()
    actual = labels.numpy()
    accuracy = 100 * accuracy_score(actual, predicted)
    matrix = confusion_matrix(actual, predicted, labels=range(5))
    return accuracy, matrix

## **5. Run all regularization studies**

In [ ]:
studies = [
    {"name": "No regularization", "file": "no_regularization"},
    {"name": "L2, lambda = 0.1", "file": "l2_0_1", "l2": 0.1},
    {"name": "L2, lambda = 1.0", "file": "l2_1_0", "l2": 1.0},
    {"name": "L2, lambda = 10.0", "file": "l2_10_0", "l2": 10.0},
    {"name": "Dropout, 10%", "file": "dropout_10", "dropout": 0.10},
    {"name": "Dropout, 20%", "file": "dropout_20", "dropout": 0.20},
    {"name": "Dropconnect, 5%", "file": "dropconnect_05", "dropconnect": 0.05},
    {"name": "Dropconnect, 10%", "file": "dropconnect_10", "dropconnect": 0.10},
    {"name": "Post-activation batch norm", "file": "batch_norm", "batch_norm": True},
]

torch.manual_seed(17)
initial_model = Network()
initial_weights = initial_model.state_dict()

In [ ]:
results = []

for study in studies:
    torch.manual_seed(17)
    model = Network(
        dropout=study.get("dropout", 0),
        dropconnect=study.get("dropconnect", 0),
        batch_norm=study.get("batch_norm", False),
    )
    model.load_state_dict(initial_weights, strict=False)
    train_error, val_error = train_model(model, study.get("l2", 0))

    train_accuracy, train_matrix = evaluate(model, x_train, y_train)
    val_accuracy, val_matrix = evaluate(model, x_val, y_val)
    test_accuracy, test_matrix = evaluate(model, x_test, y_test)

    results.append([study["name"], len(train_error), train_accuracy, val_accuracy, test_accuracy])

    fig, axes = plt.subplots(1, 4, figsize=(13, 3.1))
    axes[0].plot(train_error, label="Train")
    axes[0].plot(val_error, label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy", title="Error")
    axes[0].legend(frameon=False)

    for axis, split, accuracy, matrix in zip(
        axes[1:],
        ["Train", "Val", "Test"],
        [train_accuracy, val_accuracy, test_accuracy],
        [train_matrix, val_matrix, test_matrix],
    ):
        axis.imshow(matrix, cmap="Blues")
        for row in range(5):
            for column in range(5):
                color = "white" if matrix[row, column] > matrix.max() / 2 else "black"
                axis.text(column, row, matrix[row, column], ha="center", va="center", fontsize=7, color=color)
        axis.set(xlabel="Predicted", ylabel="True", title=f"{split}: {accuracy:.1f}%")
        axis.set_xticks(range(5))
        axis.set_yticks(range(5))

    fig.suptitle(study["name"], fontweight="bold")
    fig.tight_layout()
    fig.savefig(f"./submissions/figures/{study['file']}.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

## **6. Compare the results**

In [ ]:
results = pd.DataFrame(
    results,
    columns=["Method", "Epochs", "Train (%)", "Validation (%)", "Test (%)"],
)
results.round(2)

In [ ]:
axis = results.set_index("Method")[["Train (%)", "Validation (%)", "Test (%)"]].plot.bar(
    figsize=(11, 4), width=0.8
)
axis.set(xlabel="", ylabel="Accuracy (%)", ylim=(0, 100))
axis.legend(frameon=False, ncol=3)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("./submissions/figures/accuracy_summary.png", dpi=180, bbox_inches="tight")
plt.show()